In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path


In [3]:
pd.set_option("display.max_columns",None)
pd.set_option("display.max_rows",None)
pd.set_option("display.float_format" ,"{:,.2f}".format)
print("import all libraries sucessfully..!!")


import all libraries sucessfully..!!


In [4]:
PROJECT_ROOT = Path.cwd().parent
PROCESSED_DATA = PROJECT_ROOT / "data" / "processed"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("PROCESSED_DATA:", PROCESSED_DATA)

PROJECT_ROOT: e:\Learning\Projects\Olist-Ecom
PROCESSED_DATA: e:\Learning\Projects\Olist-Ecom\data\processed


In [5]:
if not PROCESSED_DATA.exists():
    raise FileNotFoundError("Processed data folder not found: {PROCESSED_DATA}")
else:
    print(f"File has been founded")

File has been founded


In [6]:
cleanedCSV = sorted(PROCESSED_DATA.glob("*.csv"))
print(f"Number of CSV files {len(cleanedCSV)} found!!")

for file in cleanedCSV:
    print(file.name)

Number of CSV files 11 found!!
category_translation.csv
customers.csv
geolocation.csv
master_order_items.csv
master_orders.csv
order_items.csv
order_payments.csv
order_reviews.csv
orders.csv
products.csv
sellers.csv


In [7]:
# DATASET CONFIGURATION CREATION
DATA_FILES = {
    "customers": "customers.csv",
    "geolocation": "geolocation.csv",
    "order_items": "order_items.csv",
    "order_payments": "order_payments.csv",
    "order_reviews": "order_reviews.csv",
    "orders": "orders.csv",
    "products": "products.csv",
    "sellers": "sellers.csv",
    "category_translation": "category_translation.csv",
    "master_orderItems": "master_order_items.csv",
    "master_order": "master_orders.csv"
}

print("Dataset configuration created.")

Dataset configuration created.


In [8]:
dataset = {}

for name, fileName in DATA_FILES.items():
    filePath = PROCESSED_DATA / fileName 
    if not filePath.exists():
        raise FileNotFoundError(f"{fileName} is not found")
    dataset[name] = pd.read_csv(filePath)

print("All datasets loaded Successfully!!!")

    # print(filePath.exists())

# PROCESSED_DATA

All datasets loaded Successfully!!!


In [9]:
masterOrder = dataset['master_order']
customers = dataset['customers']
products = dataset['products']
orderItems = dataset['order_items']
sellers = dataset['sellers']


Business KPI

In [20]:
totalOrders = masterOrder['order_id'].nunique()
totalCustomers = customers['customer_unique_id'].nunique()
totalSellers = sellers["seller_id"].nunique()
totalProducts = products["product_id"].nunique()
totalRevenue = (orderItems['item_total_value'].sum()).round(2)
orderRevenue = (orderItems.groupby('order_id')['item_total_value'].sum())
averageOrderValue = (orderRevenue.mean()).round(2)
totalCities = masterOrder['customer_city'].nunique()
totalStates = masterOrder['customer_state'].nunique()

print(f"Total Orders: {totalOrders}")
print(f"Total Customers: {totalCustomers}")
print(f"Total Sellers: {totalSellers}")
print(f"Total Products: {totalProducts}")
print(f"Total Revenue: {totalRevenue}")
print(f"Averge Order Value: {averageOrderValue}")
print(f"Total City: {totalCities}")
print(f"Total State: {totalStates}")


Total Orders: 99441
Total Customers: 96096
Total Sellers: 3095
Total Products: 32951
Total Revenue: 15843553.24
Averge Order Value: 160.58
Total City: 4119
Total State: 27


In [ ]:
OrderStatus = masterOrder['order_status'].value_counts().reset_index()
OrderStatus.columns = ['order_status', 'order_count']


,order_status,order_count
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2


In [12]:
masterOrder.columns

Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date',
       'delivery_days', 'delivery_delay_days', 'is_late', 'delivery_status',
       'customer_state', 'customer_city', 'total_price', 'total_freight',
       'total_value', 'item_count', 'review_score'],
      dtype='str')

In [13]:
deliveryStatusLoss = masterOrder.groupby("delivery_status").agg(
    totalOrder = ('order_id', 'count'),
    totalValue = ('total_value', 'sum'),
    totalItem = ('item_count', 'sum')
    
).sort_values('totalValue', ascending=False).reset_index()

deliveryStatusLoss

,delivery_status,totalOrder,totalValue,totalItem
0,On Time,88644,"14,066,769.87","101,475.00"
1,Late,7826,"1,351,624.96","8,714.00"
2,Not Delivered,2971,"425,158.41","2,461.00"


In [14]:
deliveryStatusLoss['orderPct'] = ((
    deliveryStatusLoss['totalOrder'] / deliveryStatusLoss['totalOrder'].sum()
) * 100).round(2)

deliveryStatusLoss

,delivery_status,totalOrder,totalValue,totalItem,orderPct
0,On Time,88644,"14,066,769.87","101,475.00",89.14
1,Late,7826,"1,351,624.96","8,714.00",7.87
2,Not Delivered,2971,"425,158.41","2,461.00",2.99


In [15]:
deliveryStatusLoss.drop(columns = 'order_pct', inplace=True)

KeyError: "['order_pct'] not found in axis"

In [ ]:
deliveryStatusLoss

,delivery_status,totalOrder,totalValue,totalItem,orderPct
0,On Time,88644,"14,066,769.87","101,475.00",89.14
1,Late,7826,"1,351,624.96","8,714.00",7.87
2,Not Delivered,2971,"425,158.41","2,461.00",2.99


In [ ]:
cityOrdersRevenue = masterOrder.groupby('customer_city').agg(
    totalOrder = ('order_id','count'),
    totalItem = ('item_count', 'sum'),
    totalValue = ('total_value', 'sum'),
    
).sort_values('totalOrder', ascending = False).head(10).reset_index()

,totalOrder,totalItem,totalValue
customer_city,,,
sao paulo,15540,"17,808.00","2,170,227.12"
rio de janeiro,6882,"7,837.00","1,154,234.02"
belo horizonte,2773,"3,144.00","416,733.39"
brasilia,2131,"2,392.00","352,305.14"
curitiba,1521,"1,751.00","244,739.87"
campinas,1444,"1,654.00","212,541.70"
porto alegre,1379,"1,612.00","224,064.09"
salvador,1245,"1,412.00","216,772.40"
guarulhos,1189,"1,329.00","163,575.82"


In [ ]:
# masterOrder.loc[(masterOrder['customer_city'] == 'sao bernardo do campo'), 'item_count']
masterOrder.loc[(masterOrder['customer_city'] == 'sao bernardo do campo'), 'item_count'].sum()

np.float64(1060.0)